# Synthetic Model Training Dry Run

This notebook tests the planned modelling pipeline using synthetic data only.

These outputs are not study results and will not be used in the final analysis.

In [1]:
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42

In [2]:
rng = np.random.default_rng(RANDOM_STATE)

n_samples = 2000
n_mut_features = 50
n_cna_features = 50

X_mut = rng.binomial(
    1,
    0.10,
    size=(n_samples, n_mut_features)
)

X_cna = rng.binomial(
    1,
    0.15,
    size=(n_samples, n_cna_features)
)

# Add some artificial signal so models have something to learn
score = (
    1.2 * X_mut[:, 0]
    + 0.8 * X_mut[:, 1]
    + 1.5 * X_cna[:, 0]
    + 1.0 * X_cna[:, 1]
    + rng.normal(0, 1, n_samples)
)

y = (score > np.median(score)).astype(int)

print("Stage 1–3:", (y == 0).sum())
print("Stage 4:", (y == 1).sum())

Stage 1–3: 1000
Stage 4: 1000


In [3]:
X_mut = pd.DataFrame(
    X_mut,
    columns=[f"MUT_gene_{i}" for i in range(n_mut_features)]
)

X_cna = pd.DataFrame(
    X_cna,
    columns=[f"CNA_gene_{i}" for i in range(n_cna_features)]
)

X_combined = pd.concat([X_mut, X_cna], axis=1)

print("Mutations:", X_mut.shape)
print("CNA:", X_cna.shape)
print("Combined:", X_combined.shape)

Mutations: (2000, 50)
CNA: (2000, 50)
Combined: (2000, 100)


In [4]:
indices = np.arange(n_samples)

train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    stratify=y[temp_idx],
    random_state=RANDOM_STATE
)

print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

Train: 1400
Validation: 300
Test: 300


In [ ]:
#Defne model families